# Placing MERFISH cells on a Visium H&E image

When one side is an image there is nothing to rasterize it into, so the *other* side is
rasterized instead and `align_stalign_image` fits image to image.

The two live in different units -- microns for the MERFISH section, pixels for the H&E -- and
neither is restated anywhere: each element carries its own placement and the solver reads the
units off it. Upstream's equivalent is `merfish-visium-alignment`.

## Inputs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, spatialdata as sd
from spatialdata.models import Image2DModel, PointsModel
from squidpy.experimental.im import rasterize_points

MERFISH = ('merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase'
           '_Slice2_Replicate3_cell_metadata_S2R3.csv.gz')
cells = pd.read_csv(MERFISH)
xy = np.c_[cells['center_x'], cells['center_y']].astype(float)

he = plt.imread('visium_data/tissue_hires_image.png')[..., :3]
visium = sd.SpatialData(images={'he': Image2DModel.parse(
    np.moveaxis(he, -1, 0).astype(float), dims=('c', 'y', 'x'))})

merfish = sd.SpatialData(points={'cells': PointsModel.parse(xy)})
rasterize_points(merfish, 'cells', dx=30.0, blur=1.0, key_added='section')
print(f'{len(xy)} cells rasterized to {tuple(np.asarray(merfish["section"]).shape)}, '
      f'H&E is {he.shape}')

Landmarks picked on these two, five named regions with several points each. Row order is the
correspondence, so both sides are flattened in the same key order -- and each side's points are
in that side's own units, microns against pixels.

In [ ]:
picked = {side: np.load(f'visium_data/{name}_points.npy', allow_pickle=True).item()
          for side, name in (('query', 'Merfish_S2_R3'), ('ref', 'tissue_hires_image'))}
regions = list(picked['ref'])
paired = {side: np.array([p for r in regions for p in picked[side][r]], dtype=float)
          for side in picked}
print(f'{len(paired["ref"])} pairs over {len(regions)} regions: {", ".join(regions)}')

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
ax[0].scatter(*xy.T, s=0.4, alpha=0.15); ax[0].scatter(*paired['query'].T, s=30, c='red')
ax[0].set_title('MERFISH section, in microns'); ax[0].invert_yaxis(); ax[0].set_aspect('equal')
ax[1].imshow(he); ax[1].scatter(*paired['ref'].T, s=30, c='red')
ax[1].set_title('Visium H&E, in pixels')
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## The fit

Upstream's own solver values for this pair. `sigmaP` weights the landmark matching term, which
matters more here than in the point-cloud case: the two modalities do not share an intensity
scale, so the landmarks carry much of the correspondence.

In [ ]:
from squidpy.experimental.tl import align_stalign_image

fit = align_stalign_image(
    visium, merfish, image_key=('he', 'section'),
    landmarks_ref=paired['ref'], landmarks_query=paired['query'],
    niter=200, sigmaM=0.2, sigmaB=0.19, sigmaA=0.3, sigmaP=2e-1,
    epL=5e-11, epT=5e-4, epV=5e1,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

## Every cell, placed on the image

In [ ]:
placed = np.asarray(fit.transform(xy))
moved_landmarks = np.asarray(fit.transform(paired['query']))
residual = np.linalg.norm(moved_landmarks - paired['ref'], axis=1)
print(f'landmark residual after fitting: median {np.median(residual):.1f} px, '
      f'worst {residual.max():.1f} px')

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(he); ax[0].set_title('Visium H&E')
ax[1].imshow(he)
ax[1].scatter(*placed.T, s=0.4, alpha=0.15, c='tab:blue')
ax[1].scatter(*paired['ref'].T, s=30, c='red', label='target landmarks')
ax[1].set_title('MERFISH cells placed on it'); ax[1].legend(fontsize=8)
for a in ax:
    a.set_xticks([]); a.set_yticks([])